# QTrans—ST-AWFD D2 公平嵌套调参与冻结

本 Notebook 下载 STMicroelectronics 官方 D2 数据，按 `MaterialID` 聚合，并在不查看外层测试折的条件下完成等预算嵌套调参。它只产生验证汇总和 `frozen_nested_selection.json`，不产生正式测试结论。

## 监督式任务边界

D2 的出版方训练群体只有正常晶圆，原论文研究的是无监督异常检测。为了进行监督式、参数匹配的 QTrans 比较，本实验只使用出版方 `is_test=1` 的评估群体（其中同时存在正常与异常晶圆），并在该群体内部重新建立严格嵌套交叉验证。出版方 `is_test=0` 的正常晶圆不参与，避免标签与来源群体混杂。该结果不能与原论文的无监督 OCSVM 数字直接比较。

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import numpy as np
import pandas as pd
import torch
import qcs_balanced_nested_tuning as nested_module

from qcs_st_awfd_d2 import (
    ST_AWFD_D2_SHA256, ST_AWFD_D2_URL, load_st_awfd_d2,
    make_st_awfd_d2_folds, st_awfd_d2_supervised_cohort,
)
from qcs_balanced_nested_tuning import (
    candidate_table, epoch_diagnostics, expected_jobs,
    finalize_nested_selection, nested_parameter_audit,
    run_nested_search, tuning_progress, tuning_summary,
)

EXPECTED_TUNING_VERSION = '2026-08-29-st-awfd-d2-v2'
actual_version = getattr(nested_module, 'NESTED_TUNING_CODE_VERSION', None)
if actual_version != EXPECTED_TUNING_VERSION:
    raise RuntimeError(
        '服务器 qcs_balanced_nested_tuning.py 不是 ST-AWFD D2 新版。'
        '请上传 qcs_st_awfd_d2.py、qcs_balanced_nested_tuning.py 和 '
        'qcs_balanced_nested_formal.py，重启 Kernel 后从头运行。'
    )
pd.set_option('display.max_columns', 100)
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_st_awfd_d2.py').exists():
    raise RuntimeError('请从 /root/xxx/autodl 目录打开本 Notebook')
DATA_DIR = PROJECT_DIR / 'data' / 'raw' / 'ST-AWFD_D2'
D2_PATH = DATA_DIR / 'D2.zip'
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'balanced_binary_nested_tuning'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
display(pd.DataFrame([{'device': str(DEVICE), 'data': str(D2_PATH), 'artifacts': str(ARTIFACT_ROOT)}]))

## 1. 下载并锁定官方 D2.zip

下载地址来自 STMicroelectronics 官方 GitHub 仓库。文件哈希不匹配时立即停止，不继续训练。

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not D2_PATH.exists():
    temporary = D2_PATH.with_suffix('.zip.part')
    print('Downloading:', ST_AWFD_D2_URL)
    urlretrieve(ST_AWFD_D2_URL, temporary)
    temporary.replace(D2_PATH)
digest = hashlib.sha256(D2_PATH.read_bytes()).hexdigest()
if digest != ST_AWFD_D2_SHA256:
    raise RuntimeError(f'D2.zip SHA-256 不匹配：{digest}')
print('D2.zip SHA-256:', digest)

## 2. 晶圆级数据与泄漏审计

每个 `MaterialID` 是一个样本。两个工艺步骤、20个测量变量分别计算 mean/std/min/max，得到160维向量。任何时间行都不会被当成独立样本。

In [ ]:
d2 = load_st_awfd_d2(DATA_DIR)
cohort = st_awfd_d2_supervised_cohort(d2)
balanced, outer_folds = make_st_awfd_d2_folds(
    d2.y, eligible_indices=cohort, balance_seed=2026, split_seed=4096
)
audit = pd.DataFrame([{
    'all_material_ids': len(d2.y),
    'source_evaluation_cohort': len(cohort),
    'cohort_normal': int((d2.y[cohort] == 0).sum()),
    'cohort_abnormal': int((d2.y[cohort] == 1).sum()),
    'balanced_material_ids': len(balanced),
    'aggregated_features': d2.x.shape[1],
    'outer_folds': len(outer_folds),
    'mixed_source_split_materials': d2.mixed_source_split_materials,
}])
display(audit)
assert len(balanced) == 474
assert np.bincount(d2.y[balanced], minlength=2).tolist() == [237, 237]

## 3. 候选空间与参数量审计

四个模型各有8个候选配置；每个候选ID下，四个模型相对量子模型的参数量差必须小于1%。候选空间不会根据部分结果追加。

In [ ]:
display(candidate_table())
parameter_audit = nested_parameter_audit(160)
display(parameter_audit)
print('预期调参任务数：', expected_jobs('st_awfd_d2'))

## 4. 五外折公平嵌套调参

总任务数为 `5外折 × 3内折 × 4模型 × 8候选 = 480`。首次环境检查可将 `MAX_JOBS=1`，确认成功后改回 `None` 完成所有剩余任务。

In [ ]:
before = tuning_progress(ARTIFACT_ROOT, 'st_awfd_d2')
display(before.groupby(['outer_fold', 'model'])['complete'].agg(['sum', 'count']))
print(f"已完成 {int(before.complete.sum())}/{len(before)}")

In [ ]:
MAX_JOBS = None
validation_results = run_nested_search(
    dataset='st_awfd_d2', data_dir=DATA_DIR, artifact_dir=ARTIFACT_ROOT,
    balance_seed=2026, outer_seed=4096, inner_seed=8192,
    max_jobs=MAX_JOBS, device=DEVICE,
)
print(f'当前验证结果 {len(validation_results)}/480')
display(tuning_summary(validation_results).groupby(['outer_fold', 'model']).head(2))

## 5. 完整性检查并冻结

只有480个任务全部完成后才能生成冻结文件。选择规则固定为 `平均验证 Macro-F1 - 0.25 × 标准差`。

In [ ]:
after = tuning_progress(ARTIFACT_ROOT, 'st_awfd_d2')
if not after.complete.all():
    raise RuntimeError(f'调参未完成：{int(after.complete.sum())}/480')
validation_summary, frozen = finalize_nested_selection(
    ARTIFACT_ROOT, 'st_awfd_d2'
)
display(pd.DataFrame(frozen['selections']))
display(epoch_diagnostics(validation_results).query("model == 'quantum_transformer'"))
print('冻结文件：', ARTIFACT_ROOT / 'st_awfd_d2' / 'frozen_nested_selection.json')

## 输出位置

```text
artifacts/balanced_binary_nested_tuning/st_awfd_d2/
├── manifest.json
├── nested_indices.npz
├── parameter_audit.csv
├── validation_results.csv
├── validation_summary.csv
└── frozen_nested_selection.json
```

冻结后不要再根据正式结果修改候选配置、聚合方式、样本群体或随机种子。